In [ ]:
import pandas as pd
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, when, count

In [ ]:
data_path = "C:/GitHub/telecom_churn_predictor/data/train.csv"
spark = SparkSession.builder.appName("telecomChurn").getOrCreate()

df = spark.read.csv(data_path, header=True, inferSchema=True)

df.printSchema()
df.show(5)
print(f"Number of rows: {df.count()}")
print(f"Number of columns: {len(df.columns)}")
df.groupBy("churn_probability").count().show()

In [ ]:
# Set a threshold for null percentage (e.g., 30%)
# Select columns where the percentage of null values exceeds the threshold
null_percentage_threshold = 30
null_count = df.select([(count(when(col(c).isNull(), c)) / df.count() * 100).alias(c) 
                       for c in df.columns])
# null_count.show()

# Convert the null_count DataFrame to a dictionary and filter columns based 
# on the threshold
cols_to_drop = [k for k, v in null_count.collect()[0].asDict().items() 
                if v > null_percentage_threshold]
# print(f"Columns to drop (more than {null_percentage_threshold}% null values): {cols_to_drop}")
df = df.drop(*cols_to_drop)

# Remove columns that are not relevant for churn prediction
cols_to_remove = ['id', 'circle_id', 'last_date_of_month_6', 
                  'last_date_of_month_7', 'last_date_of_month_8',
                  'date_of_last_rech_6', 'date_of_last_rech_7', 'date_of_last_rech_8',
                  'date_of_last_rech_data_6', 'date_of_last_rech_data_7', 'date_of_last_rech_data_8']
df = df.drop(*cols_to_remove)

df = df.fillna(0)
print(f"number of cols after cleaning: {len(df.columns)}")

# Make sure after cleaning, features we want are undamaged
df.filter(df.churn_probability == 1).count()

In [ ]:
# Generate new features based on existing ones
df = df.withColumn("arpu_trend", col("arpu_8") - col("arpu_6"))
df = df.withColumn("mou_trend", col("total_og_mou_8") - col("total_og_mou_6"))
df = df.withColumn("rech_trend", col("total_rech_amt_8") - col("total_rech_amt_6"))

df.select("arpu_trend", "mou_trend", "rech_trend").show(5)

In [ ]:
# Convert the Spark DataFrame to a Pandas DataFrame for 
# further analysis or modeling (e.g., with scikit-learn)
df_pd = df.toPandas()
print(df_pd.shape)
df_pd.to_csv("C:/GitHub/telecom_churn_predictor/data/processed_data.csv", index=False)
print("Data preprocessing completed and saved to processed_data.csv")